In [1]:
# Copyright (c) 2026 icurate.cc
# Licensed under the MIT License
import json

# Path to COCO annotations file
ann_path = "/app/coco/annotations/instances_train2017.json"  # adjust if needed

# Load JSON
with open(ann_path, "r") as f:
    data = json.load(f)

# Extract categories
categories = data["categories"]

# Build mappings
id_to_name = {cat["id"]: cat["name"] for cat in categories}
name_to_id = {cat["name"]: cat["id"] for cat in categories}

# Print categories
print(f"Total categories: {len(categories)}\n")

for cat in categories:
    print(f'ID: {cat["id"]:2d}  Name: {cat["name"]}')

Total categories: 80

ID:  1  Name: person
ID:  2  Name: bicycle
ID:  3  Name: car
ID:  4  Name: motorcycle
ID:  5  Name: airplane
ID:  6  Name: bus
ID:  7  Name: train
ID:  8  Name: truck
ID:  9  Name: boat
ID: 10  Name: traffic light
ID: 11  Name: fire hydrant
ID: 13  Name: stop sign
ID: 14  Name: parking meter
ID: 15  Name: bench
ID: 16  Name: bird
ID: 17  Name: cat
ID: 18  Name: dog
ID: 19  Name: horse
ID: 20  Name: sheep
ID: 21  Name: cow
ID: 22  Name: elephant
ID: 23  Name: bear
ID: 24  Name: zebra
ID: 25  Name: giraffe
ID: 27  Name: backpack
ID: 28  Name: umbrella
ID: 31  Name: handbag
ID: 32  Name: tie
ID: 33  Name: suitcase
ID: 34  Name: frisbee
ID: 35  Name: skis
ID: 36  Name: snowboard
ID: 37  Name: sports ball
ID: 38  Name: kite
ID: 39  Name: baseball bat
ID: 40  Name: baseball glove
ID: 41  Name: skateboard
ID: 42  Name: surfboard
ID: 43  Name: tennis racket
ID: 44  Name: bottle
ID: 46  Name: wine glass
ID: 47  Name: cup
ID: 48  Name: fork
ID: 49  Name: knife
ID: 50  Name:

In [2]:
from super_gradients.training import models

NUM_CLASSES = len(categories)
model = models.get(
    "yolo_nas_s",
    num_classes=NUM_CLASSES,
    pretrained_weights=None  # 🔥 key line
)

The console stream is logged into /root/sg_logs/console.log


/usr/local/lib/python3.10/dist-packages/super_gradients/common/environment/cfg_utils.py:6: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  import pkg_resources
[2026-03-23 21:35:16] INFO - crash_tips_setup.py - Crash tips is enabled. You can set your environment variable to CRASH_HANDLER=FALSE to disable it


[WARNING]No module named 'pycocotools'


In [3]:
from super_gradients.training import dataloaders

train_loader = dataloaders.get(
    "coco2017_train",
    dataset_params={
        "data_dir": "/app/coco",
        "json_file": "instances_train2017.json"  # ⚠️ only filename
    },
    dataloader_params={
        "batch_size": 8,
        "num_workers": 1
    }
)

val_loader = dataloaders.get(
    "coco2017_val",
    dataset_params={
        "data_dir": "/app/coco",
        "json_file": "instances_val2017.json"
    },
    dataloader_params={
        "batch_size": 8,
        "num_workers": 1
    }
)

[2026-03-23 21:35:34] INFO - detection_dataset.py - Dataset Initialization in progress. `cache_annotations=True` causes the process to take longer due to full dataset indexing.
Indexing dataset annotations: 100%|██████████████████████████████████████████| 118287/118287 [00:03<00:00, 36194.07it/s]
[2026-03-23 21:35:38] INFO - detection_dataset.py - Dataset Initialization in progress. `cache_annotations=True` causes the process to take longer due to full dataset indexing.
Indexing dataset annotations: 100%|██████████████████████████████████████████████| 5000/5000 [00:00<00:00, 35904.36it/s]


In [4]:
train_losses = []
val_losses = []
epochs = []

def log_metrics(context):
    if context.phase == "train":
        train_losses.append(context.metrics_dict.get("Loss", None))
    elif context.phase == "valid":
        val_losses.append(context.metrics_dict.get("Loss", None))
        epochs.append(context.epoch)

In [13]:
from super_gradients.training import Trainer
from super_gradients.training.losses import PPYoloELoss

trainer = Trainer(
    experiment_name="yolo_nas_coco",
    ckpt_root_dir="/app/checkpoints"
)
# YOLO-NAS uses PPYoloELoss for training
loss_fn = PPYoloELoss(
    num_classes=NUM_CLASSES,
    use_varifocal_loss=True
)

trainer.train(
    model=model,
    training_params={
        "max_epochs": 50,
        "save_ckpt_epoch_list": list(range(0, 51, 10)),  # every 10 epochs
        "initial_lr": 1e-3,
        "lr_mode": "cosine",
        "optimizer": "AdamW",
        "batch_size": 8,
        "metric_to_watch": "PPYoloELoss/loss",  # monitor total loss
        "loss": loss_fn  # instance of YoloXFastDetectionLoss
    },
    train_loader=train_loader,
    valid_loader=val_loader
)

[2026-03-23 21:44:43] INFO - sg_trainer.py - Starting a new run with `run_id=RUN_20260323_214443_540077`
[2026-03-23 21:44:43] INFO - sg_trainer.py - Checkpoints directory: /app/checkpoints/yolo_nas_coco/RUN_20260323_214443_540077


The console stream is now moved to /app/checkpoints/yolo_nas_coco/RUN_20260323_214443_540077/console_Mar23_21_44_43.txt


[2026-03-23 21:44:47] INFO - sg_trainer_utils.py - TRAINING PARAMETERS:
    - Mode:                         Single GPU
    - Number of GPUs:               1          (1 available on the machine)
    - Full dataset size:            117266     (len(train_set))
    - Batch size per GPU:           8          (batch_size)
    - Batch Accumulate:             1          (batch_accumulate)
    - Total batch size:             8          (num_gpus * batch_size)
    - Effective Batch size:         8          (num_gpus * batch_size * batch_accumulate)
    - Iterations per epoch:         14658      (len(train_loader))
    - Gradient updates per epoch:   14658      (len(train_loader) / batch_accumulate)
    - Model: YoloNAS_S  (19.05M parameters, 19.05M optimized)
    - Learning Rates and Weight Decays:
      - default: (19.05M parameters). LR: 0.001 (19.05M parameters) WD: 0.01, (19.05M parameters)

[2026-03-23 21:44:47] INFO - sg_trainer.py - Started training for 200 epochs (0/199)

Train epoch 0:

In [12]:
trainer.stop_training = True